# Module 25 — AI Observability Platform

Instrument an end-to-end agentic system and connect execution telemetry to quality, cost, latency, security and debugging.

In [ ]:
from dataclasses import dataclass, field
@dataclass
class Span:
    id:str; trace:str; parent:str|None; tenant:str; component:str; start:float; end:float; attrs:dict=field(default_factory=dict)
    @property
    def ms(self): return (self.end-self.start)*1000


## 1. End-to-end trace
Model an agent → retrieval → model → MCP tool → verifier trajectory.

In [ ]:
spans=[Span('root','r1',None,'t1','agent',0,1),Span('retr','r1','root','t1','retrieval',1,1.4,{'cost_usd':.01}),Span('model','r1','root','t1','model',1.4,3,{'tokens':500,'cost_usd':.12}),Span('tool','r1','model','t1','mcp',3,4,{'cost_usd':.03}),Span('verify','r1','root','t1','verifier',4,4.3)]
[(s.component,s.ms) for s in spans]


## 2. Critical path
Calculate the longest causal execution chain. Extend the exercise to parallel worker branches.

In [ ]:
print('critical-path approximation:',sum(s.ms for s in spans),'ms')


## 3. Cost attribution
Aggregate model and tool costs by component and calculate cost per successful task.

In [ ]:
cost={}; tokens={}
for s in spans:
  cost[s.component]=cost.get(s.component,0)+s.attrs.get('cost_usd',0)
  tokens[s.component]=tokens.get(s.component,0)+s.attrs.get('tokens',0)
print(cost,tokens)


## 4. Quality correlation
Add retrieval Recall@K, groundedness and verifier-pass signals. Compare a good run with a degraded retrieval run.

In [ ]:
runs=[{'retrieval_recall':.95,'groundedness':.94,'success':1},{'retrieval_recall':.55,'groundedness':.61,'success':0}]
runs


## 5. Retry storm
Inject repeated failed tool spans. Calculate retry amplification and define an alert threshold.

In [ ]:
attempts=[1,2,3,4,5]
print('retry amplification:',len(attempts)-1)


## 6. Loop detection
Create repeated state/tool sequences and detect when the agent is cycling before exhausting its budget.

In [ ]:
sequence=['search','read','search','read','search','read']
print('cycle detected:',len(sequence)>=4 and sequence[-2:]==sequence[-4:-2])


## 7. Security telemetry
Inject policy.denied, injection.detected and cross-tenant events. Verify security events remain visible even if ordinary traces are sampled.

## 8. Tenant-safe observability
Attempt to query another tenant's trace. Expected: deny at the telemetry query boundary.

In [ ]:
request_tenant='t1'; trace_tenant='t2'
assert request_tenant!=trace_tenant
print('cross-tenant query must be rejected')


## 9. Failure injection
Break parent IDs, timestamps, model versions, costs and security events. Measure trace completeness and identify which failures must page an operator.

## 10. Sampling
Compare full tracing with head sampling and tail sampling. Preserve all security incidents and failed high-value runs.

## 11. Alert engineering
Design alerts for task-success SLO, p95 latency, cost/run, retry amplification, verifier failure and suspicious tool sequences. Avoid alerting on every individual model error.

## 12. Module integration
Feed telemetry into Module 22's causal debugger. Add Module 26 evaluation outcomes to the same trace ID.

# 15 extension exercises
1. Build a trace validator.
2. Build a span-tree visualizer.
3. Implement critical-path calculation.
4. Add p50/p95/p99 metrics.
5. Add cost attribution.
6. Detect retry storms.
7. Detect agent loops.
8. Detect anomalous tool sequences.
9. Implement tenant-safe queries.
10. Implement secret redaction.
11. Implement tail-based sampling simulation.
12. Build SLO evaluation.
13. Build incident alerts.
14. Correlate quality with infrastructure metrics.
15. Build the AegisAI Observability Platform gold challenge.

# Gold challenge
Build an observability control plane for Modules 18–24 that supports distributed traces, causal debugging, cost/latency/quality analysis, security investigation and evaluation-ready telemetry.